# Part 1 — Explore the MOJ sales extract

In [ ]:
import re
import pandas as pd
import numpy as np
from hijridate import Hijri

In [ ]:
# change the seeting of the cells to show the entire content of coulmn ( eexpand the size of coulmn)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
df = pd.read_excel("moj_sales_extract.xlsx")
df.head()

In [ ]:
print("rows, columns:", df.shape)

In [ ]:
print(df.dtypes)

In [ ]:
df.tail(3)

In [ ]:
# Print a concise summary of a DataFrame
df.info()

In [ ]:
# number of non-NA values
df.count()

In [ ]:
#Generates descriptive statistics of the numeric columns
df.describe()

In [ ]:
# Retrieve list of columns
df.columns

## 1.Missing values

In [ ]:
df.isna().sum()

## 2. Duplicates
SERIAL should identify one real estate record, so it should be unique.

In [ ]:
print("number of the identical rows:", df.duplicated().sum())
print("number of rows with identical SERIAL:", df["SERIAL"].duplicated().sum())
print("number of rows with missing SERIAL:", df["SERIAL"].isna().sum())

In [ ]:
# Same SERIAL but different content = conflicting versions of a record
dup_serials = df[df["SERIAL"].duplicated(keep=False) & df["SERIAL"].notna()]
conflicting = dup_serials.drop_duplicates().groupby("SERIAL").size()
print("SERIALs that appear with more than one distinct version:", (conflicting > 1).sum())

# Same everything except SERIAL = possible re-keyed duplicates
cols_wo_serial = [c for c in df.columns if c != "SERIAL"]
print("Rows identical on every column except SERIAL:", df.duplicated(subset=cols_wo_serial).sum())

## 3. Dates (Hijri)
First check what shapes the text takes, then parse and convert to Gregorian.

In [ ]:
raw = df["DEED_DATE_H"]
print("dtype:", raw.dtype)

In [ ]:
# Replace every digit with 9 so we see the *shape* of each value
shapes = raw.astype("string").str.replace(r"\d", "9", regex=True)
shapes.value_counts(dropna=False).head(15)

In [ ]:
def parse_hijri(s):
    if pd.isna(s):
        return pd.NaT
    m = re.match(r"^\s*(\d{4})\D+(\d{1,2})\D+(\d{1,2})\s*$", str(s))
    if not m:
        return pd.NaT
    y, mo, d = map(int, m.groups())
    try:
        return pd.Timestamp(Hijri(y, mo, d).to_gregorian())
    except (ValueError, OverflowError):
        return pd.NaT

# Convert each distinct date once (fast)
uniq = raw.dropna().unique()
mapping = {u: parse_hijri(u) for u in uniq}
df["date_g"] = raw.map(mapping)

print("Missing date text:", raw.isna().sum())
print("Present but could not be parsed / invalid:", (raw.notna() & df["date_g"].isna()).sum())
print("Gregorian range:", df["date_g"].min().date(), "to", df["date_g"].max().date())
print()
print("Hijri year values:")
print(raw.astype("string").str.extract(r"^\s*(\d{4})")[0].value_counts().sort_index())

In [ ]:
# Unparseable examples (if any)
bad = df.loc[raw.notna() & df["date_g"].isna(), "DEED_DATE_H"]
bad.value_counts().head(20)

In [ ]:
df["month_g"] = df["date_g"].dt.to_period("M")
df["quarter_g"] = df["date_g"].dt.to_period("Q")
print(df["month_g"].value_counts().sort_index())
print()
print(df["quarter_g"].value_counts().sort_index())

## 4. Coverage — city, district, property type, usage

In [ ]:
print("Distinct cities:", df["RS_CITY_NAME"].nunique())
df["RS_CITY_NAME"].value_counts(dropna=False)

In [ ]:
print("Distinct districts:", df["NEIGHBORHOOD_NAME"].nunique())
print("Missing district:", df["NEIGHBORHOOD_NAME"].isna().sum())
df["NEIGHBORHOOD_NAME"].value_counts(dropna=False).head(30)

In [ ]:
print("RS_TYPE"); print(df["RS_TYPE"].value_counts(dropna=False)); print()
print("new_use"); print(df["new_use"].value_counts(dropna=False))

In [ ]:
# The two classifications describe the same property but are filled in independently.
pd.crosstab(df["RS_TYPE"].fillna("<missing>"), df["new_use"].fillna("<missing>"))

## 5. Text hygiene — does the same name appear in several spellings?

In [ ]:
def normalise(s):
    if pd.isna(s):
        return s
    s = str(s)
    s = re.sub(r"[\u064B-\u065F\u0670]", "", s)   # diacritics
    s = s.replace("\u0640", "")                     # tatweel
    s = re.sub(r"[أإآٱ]", "ا", s)                   # alef variants
    s = s.replace("ة", "ه").replace("ى", "ي")       # ta marbuta / alef maqsura
    return re.sub(r"\s+", " ", s).strip()

for col in ["RS_CITY_NAME", "NEIGHBORHOOD_NAME", "RS_TYPE", "new_use"]:
    s = df[col].dropna().astype(str)
    padded = (s != s.str.strip()).sum()
    multi_space = s.str.contains(r"\s{2,}").sum()
    before = s.nunique()
    after = s.map(normalise).nunique()
    print(f"{col:20s} distinct={before:5d}  after normalising={after:5d}  "
          f"padded={padded}  double-spaces={multi_space}")

In [ ]:
# Which district spellings collapse into the same normalised name?
d = df["NEIGHBORHOOD_NAME"].dropna().astype(str)
g = pd.DataFrame({"raw": d, "norm": d.map(normalise)}).drop_duplicates()
clash = g.groupby("norm")["raw"].apply(list)
clash[clash.map(len) > 1].head(20)

## 6. Numbers — area, meter price, and derived transaction value
`METER_PRICE` is per square meter

**value = AREA × METER_PRICE**.

In [ ]:

for c in ["AREA", "METER_PRICE"]:
    print(c, "dtype:", df[c].dtype)
    conv = pd.to_numeric(df[c], errors="coerce")
    print("  present but non-numeric:", (df[c].notna() & conv.isna()).sum())
    print("  missing:", df[c].isna().sum(), "| zero:", (conv == 0).sum(), "| negative:", (conv < 0).sum())
    df[c] = conv

In [ ]:
df["value"] = df["AREA"] * df["METER_PRICE"]
q = [0, .001, .01, .05, .25, .5, .75, .95, .99, .999, 1]
df[["AREA", "METER_PRICE", "value"]].quantile(q)

In [ ]:
print("10 largest METER_PRICE"); print(df.nlargest(10, "METER_PRICE")[["RS_CITY_NAME","NEIGHBORHOOD_NAME","new_use","AREA","METER_PRICE","value"]]); print()
print("10 smallest positive METER_PRICE"); print(df[df["METER_PRICE"]>0].nsmallest(10, "METER_PRICE")[["RS_CITY_NAME","NEIGHBORHOOD_NAME","new_use","AREA","METER_PRICE","value"]]); print()
print("10 largest AREA"); print(df.nlargest(10, "AREA")[["RS_CITY_NAME","NEIGHBORHOOD_NAME","new_use","AREA","METER_PRICE","value"]])

In [ ]:
# Same property type mixes very different price levels?  Median price by usage
df.groupby("new_use", dropna=False)["METER_PRICE"].agg(["count", "median", "mean", "max"]).sort_values("count", ascending=False)

## 7. Impact on headline numbers
How much would each problem move a published **average price** and **total sales value**?

Extreme values (absurd prices or areas) are so large that they hide every other effect, so the table works in two layers:
1. **Raw** vs **BASE**: the base removes extreme prices and areas. This shows how much the extremes alone distort the numbers.
2. Every ordinary problem (duplicates, missing or zero price, missing or zero area) is then measured **on top of the base**, so its own effect is visible.

Read the `vs_raw` columns for the change from the raw data, and the `vs_base` columns for the effect of each problem after the extremes are gone.
The median is shown because it stays sensible even when extreme values are present.

In [ ]:
# STEP A - find "impossible" prices and areas.
# Values span many orders of magnitude, so we judge them on a log scale:
# anything more than 3 IQRs beyond the middle 50% of the data (on log10) counts as extreme.
def log_fences(s, k=3):
    logs = np.log10(s[s > 0].dropna())
    q1, q3 = logs.quantile([.25, .75])
    return 10 ** (q1 - k * (q3 - q1)), 10 ** (q3 + k * (q3 - q1))

price_lo, price_hi = log_fences(df["METER_PRICE"])
area_lo, area_hi = log_fences(df["AREA"])

print(f"METER_PRICE plausible range: {price_lo:,.0f} to {price_hi:,.0f}")
print(f"AREA plausible range:        {area_lo:,.0f} to {area_hi:,.0f}")

extreme_price = (df["METER_PRICE"] > 0) & ~df["METER_PRICE"].between(price_lo, price_hi)
extreme_area = (df["AREA"] > 0) & ~df["AREA"].between(area_lo, area_hi)
print("rows with extreme price:", extreme_price.sum())
print("rows with extreme area: ", extreme_area.sum())
print("rows with either:       ", (extreme_price | extreme_area).sum())

# STEP B - headline numbers for a set of rows
def headline(d, label):
    ok = (d["METER_PRICE"] > 0) & (d["AREA"] > 0)      # rows where value can be computed
    return {
        "scenario": label,
        "rows": len(d),
        "mean_price": d["METER_PRICE"].mean(),          # simple average (empty ignored, zeros included)
        "median_price": d["METER_PRICE"].median(),      # the "typical" price, not pulled by extremes
        "weighted_price": d.loc[ok, "value"].sum() / d.loc[ok, "AREA"].sum(),   # total value / total area
        "total_value": d["value"].sum(),
    }

# STEP C - raw, then the base (extremes removed), then each ordinary problem ON TOP of the base
base = df[~(extreme_price | extreme_area)]
dedupe_cols = [c for c in df.columns if c not in ("date_g", "month_g", "quarter_g", "value")]

scenarios = [
    headline(df, "raw, as received"),
    headline(base, "BASE: remove extreme price / area"),
    headline(base.drop_duplicates(subset=dedupe_cols), "base + drop exact duplicate rows"),
    headline(base[base["METER_PRICE"] > 0], "base + drop missing / zero price"),
    headline(base[base["AREA"] > 0], "base + drop missing / zero area"),
]
cleaned = base.drop_duplicates(subset=dedupe_cols)
cleaned = cleaned[(cleaned["METER_PRICE"] > 0) & (cleaned["AREA"] > 0)]
scenarios.append(headline(cleaned, "base + all three combined"))

out = pd.DataFrame(scenarios).set_index("scenario")
raw_row, base_row = out.iloc[0], out.iloc[1]
out["mean_vs_raw_%"] = (out["mean_price"] / raw_row["mean_price"] - 1) * 100
out["total_vs_raw_%"] = (out["total_value"] / raw_row["total_value"] - 1) * 100
out["mean_vs_base_%"] = (out["mean_price"] / base_row["mean_price"] - 1) * 100
out["total_vs_base_%"] = (out["total_value"] / base_row["total_value"] - 1) * 100
out

## Outliers

In [ ]:
import matplotlib.pyplot as plt

def plot_histogram(df, column_name):
    """
    Plots a histogram for a given column in a DataFrame to inspect the distribution and potential outliers.
    """
    plt.figure(figsize=(6, 4))
    df[column_name].hist(bins=30, edgecolor='black')  # Using built-in Pandas method
    plt.title(f"Histogram of {column_name}")
    plt.xlabel(column_name)
    plt.ylabel("Frequency")
    plt.show()

# Example usage:
plot_histogram(df, 'AREA')

In [ ]:
plot_histogram(df, 'METER_PRICE')